In [ ]:
from src.retrievers.hybrid import HybridRetriever
from src.retrievers.bm25 import Bm25Retriever
from src.retrievers.dense import DenseRetriever
from src.embeddings import embed_texts, DEFAULT_BI_ENCODER
from src.data_utils import load_documents, chunker
from src.generate import generate
import faiss

documents = load_documents("/home/nick/new-speedrun/data/2038.pdf")
chunks = chunker(documents, chunk_size=301, chunk_overlap=30.1)

/home/nick/new-speedrun/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

query = 'What did Darren Walsh do for Work in 2027?'
pairs = [(query, doc.page_content) for doc in chunks]
scores = reranker.predict(pairs)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 14505.99it/s]


In [8]:
ranked = sorted(zip(scores, chunks), reverse=True)
for score, doc in ranked:
    print(f"[{score:.3f}] {doc}")

[0.953] page_content='Not the day he had walked along the benches shaking
hands, the day he had told a scaffolder called Walsh that
everything they were going to build would sit on the back of
men like him, and meant it. Before that. A month before. A
Monday in September 2027, when the seven of them had sat
91' metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '/home/nick/new-speedrun/data/2038.pdf', 'total_pages': 163, 'page': 91, 'page_label': '92'}
[-2.385] page_content='something fragile.
“Thank you, sir,” said Darren Walsh.
141' metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': '/home/nick/new-speedrun/data/2038.pdf', 'total_pages': 163, 'page': 141, 'page_label': '142'}
[-2.632] page_content='CHAPTER  FIVE
My name is Darren Walsh and I am thick.
That’s the ﬁrst thing. They tell you it ﬁrst so you don’t have
to work it out on your own, which is kind of them, because I
wouldn’t have. I’d have gone round for years thinkin

In [ ]:

test_queries = [
    'Does Sterling Bronson have a mag?',
    'What did Darren Walsh do for Work in 2027?',
    'Which pub does Winston Reed regularly visit?',
    'What town is this novel set in?',
    'Who is Grace?',
    'Which chapter first introduces Cassie May?',
    'Which social or occupational class does Cassie belong to?',
    'What are cats?',
    'what are transistors?',
    'does sterling own the transistors?',
    'what are the people who own the transistors referred to?'
]


def rag_answer(query: str, k: int) -> str:
    if k < 1:
        raise ValueError("k must be at least 1")
    retrieved = dense.retrieve(query, top_k=k)
    context_chunks = [doc.page_content for doc in retrieved]
    answer = generate(query, context_chunks)
    return answer